In [6]:
import os
import time
import pandas as pd
from ytmusicapi import YTMusic

PLAYLIST_LIMIT=500
PLAYLIST_SONG_LIMIT=10000
yt = YTMusic('../headers_auth.json')

def parse_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(
        lambda x: x[0]['id'])  # TODO handle > 1 artist
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_playlist(yt, playlist_meta, print_meta=False):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    if print_meta: 
        print(pd.DataFrame.from_dict(playlist_meta, orient='index'))
    tracks = parse_tracks(track_list)
    return tracks, playlist_meta

def create_rating_playlist_subset(tracks, name, rating):
    assert rating in ('LIKE', 'DISLIKE', 'INDIFFERENT')
    filtered_tracks = tracks.loc[tracks['likeStatus'] == rating]
    video_ids = filtered_tracks['videoId'].unique().tolist()
    pl_id = yt.create_playlist(
        title=name + ' ' + rating.lower(), 
        description='generated from %s includes %s subset' % (name, rating),
        privacy_status='PRIVATE', 
        video_ids=video_ids
    )
    print('Created %s playlist with id %s' % (rating, pl_id))

In [7]:
%%time
playlists = pd.DataFrame(yt.get_library_playlists(limit=PLAYLIST_LIMIT))
print('Playlists:\n %s' % sorted(playlists.sort_values('title')['title']))

Playlists:
 ["'60s Garage Rock Nuggets", "'90s Hip Hop Party", '1950s_thumbs_up', '1960s_thumbs_up', '1970s_thumbs_up', '1980s_thumbs_up', '1990s_thumbs_up', "1993 'til Infinity", '2000s_thumbs_up', '2005s_thumbs_up', '2010_thumbs_up', '2010s Electronic like', '2011_thumbs_up', '2012_thumbs_up', '2013_thumbs_up', '2014_thumbs_up', '2015_thumbs_up', '2016_thumbs_up', '2017_thumbs_up', '2018_thumbs_up', '2018_top_50_albums', '2019_thumbs_up', '2019_top_50_albums', 'Adagio', 'All-Time Hip Hop Hits', 'Ambient Unrated Albums 2018-2019', 'Analog Grooves like', 'Beat instrumentals', 'Blue Autumn', 'Brass n chill', "California Dreamin' indifferent", "California Dreamin' like", 'Chill Indie Beats indifferent', 'Chill Indie Beats like', 'Chillwave', "Classic Rock's Greatest Hits", "Classic Rock's Greatest Hits indifferent", "Classic Rock's Greatest Hits like", 'Classic Sunshine Soul', 'Classic West Coast Hip Hop indifferent', 'Classic West Coast Hip Hop like', 'Classical piano', 'Dance radio', '

In [8]:
%%time

# Example: Group public playlists
public_playlists = {}
privacy = 'PUBLIC'
for i, p in playlists.iterrows():
    if i == 0: continue # skip giant likes playlist
    metadata = yt.get_playlist(p['playlistId'], limit=PLAYLIST_SONG_LIMIT)
    if metadata['privacy'] == privacy:
        public_playlists[p['playlistId']] = p['title']
        print('Found %s playlist named: %s' % (privacy.lower(), p['title']))

list(public_playlists.values())

Found public playlist named: '60s Garage Rock Nuggets
Found public playlist named: '90s Hip Hop Party
Found public playlist named: 1993 'til Infinity
Found public playlist named: Adagio
Found public playlist named: All-Time Hip Hop Hits
Found public playlist named: Blue Autumn
Found public playlist named: Classic Rock's Greatest Hits
Found public playlist named: Classic Sunshine Soul
Found public playlist named: classical guitar
Found public playlist named: Dreams of Fall
Found public playlist named: Essential '00s Hip Hop
Found public playlist named: Essential '90s Hip Hop
Found public playlist named: Essential Folk
Found public playlist named: Essential Proto-Metal
Found public playlist named: Gentle Acoustic Instrumentals
Found public playlist named: Music Visuals
Found public playlist named: Muted Jazz
Found public playlist named: Post-Punk Pleasures
Found public playlist named: Soulful Heartache
Found public playlist named: Sunshine Reggae
Found public playlist named: Teenage Spac

["'60s Garage Rock Nuggets",
 "'90s Hip Hop Party",
 "1993 'til Infinity",
 'Adagio',
 'All-Time Hip Hop Hits',
 'Blue Autumn',
 "Classic Rock's Greatest Hits",
 'Classic Sunshine Soul',
 'classical guitar',
 'Dreams of Fall',
 "Essential '00s Hip Hop",
 "Essential '90s Hip Hop",
 'Essential Folk',
 'Essential Proto-Metal',
 'Gentle Acoustic Instrumentals',
 'Music Visuals',
 'Muted Jazz',
 'Post-Punk Pleasures',
 'Soulful Heartache',
 'Sunshine Reggae',
 'Teenage Spaceship',
 'The New Age of Classic Rock',
 'The World of Pavement']

In [3]:
# # Example: Create Unrated and Liked Subset Playlist (single case)

# playlist_name = 'Analog Grooves'
# playlist = playlists.loc[playlists['title'] == playlist_name].iloc[0] # first match
# metadata = yt.get_playlist(playlist['playlistId'], limit=PLAYLIST_SONG_LIMIT)
# tracks, metadata = parse_playlist(yt, metadata)
# print('Selected Playlist:\n%s' % metadata)

# create_rating_playlist_subset(tracks, playlist_name, 'INDIFFERENT')
# create_rating_playlist_subset(tracks, playlist_name, 'LIKE')

In [11]:
%%time
# For each playlist, Create Unrated and Liked Subset Playlist, delete original
playlist_names = [
    'Gentle Acoustic Instrumentals',
    'Muted Jazz',
    'Post-Punk Pleasures',
    'Soulful Heartache',
    'Sunshine Reggae',
    'Teenage Spaceship',
    'The New Age of Classic Rock',
    'The World of Pavement'
 ]
for playlist_name in playlist_names:
    print('Sorting %s in to like and indifferent playlists and deleting original' % playlist_name)
    playlist = playlists.loc[playlists['title'] == playlist_name].iloc[0] # first match
    metadata = yt.get_playlist(playlist['playlistId'], limit=PLAYLIST_SONG_LIMIT)
    tracks, metadata = parse_playlist(yt, metadata)
    create_rating_playlist_subset(tracks, playlist_name, 'INDIFFERENT')
    time.sleep(20)
    create_rating_playlist_subset(tracks, playlist_name, 'LIKE')
    time.sleep(20)
    yt.delete_playlist(playlist['playlistId']) 

Sorting Gentle Acoustic Instrumentals in to like and indifferent playlists and deleting original
Created INDIFFERENT playlist with id PLWptjpDqazOwqXABMqmu3oKe1UTWlMJW_
Created LIKE playlist with id PLWptjpDqazOyXU95_rii7x15McWnoex5F
Sorting Muted Jazz in to like and indifferent playlists and deleting original
Created INDIFFERENT playlist with id PLWptjpDqazOxDEtpAksZB64orqZ3HgfE-
Created LIKE playlist with id PLWptjpDqazOyzIGV9MKweZ3ylIwi1TEFr
Sorting Post-Punk Pleasures in to like and indifferent playlists and deleting original
Created INDIFFERENT playlist with id PLWptjpDqazOxKAbD10puIRfPSPw7isWbR
Created LIKE playlist with id PLWptjpDqazOxy1LbTYUzfiNjPTJuV-KCU
Sorting Soulful Heartache in to like and indifferent playlists and deleting original
Created INDIFFERENT playlist with id PLWptjpDqazOzHC1bT-lew-9zfBidMrEYq
Created LIKE playlist with id PLWptjpDqazOxJltV40sL5TWXRO4qK7RBH
Sorting Sunshine Reggae in to like and indifferent playlists and deleting original
Created INDIFFERENT pl